# [실습] 나만의 캐릭터 말투 Fine-Tuning

앞서 조선시대 신하 말투를 학습시켜 봤습니다.  
이번에는 **나만의 캐릭터 말투** 데이터를 직접 만들고 fine-tuning 해봅니다.

| 항목 | 데모에서 한 것 | 이번 실습 |
|------|----------------|----------|
| 데이터 | 조선시대 말투 200개 (제공) | 나만의 캐릭터 말투 30개 이상 (직접 제작) |
| 모델 | Qwen2.5-1.5B-Instruct | Qwen2.5-1.5B-Instruct (동일) |
| 방법 | QLoRA (Unsloth) | QLoRA (Unsloth) (동일) |

### 캐릭터 아이디어 예시

| 캐릭터 | 말투 특징 | 예시 |
|--------|----------|------|
| 부산 사투리 | ~다 아이가, ~하이소 | "밥 묵었나? 안 묵었으면 같이 가자 아이가" |
| 츤데레 | 관심 없는 척, ~별로야 | "딱히 네가 걱정돼서 그런 건 아니거든..." |
| 할머니 | ~하렴, ~구나 | "어이고 우리 손주, 밥은 먹었니? 이것도 먹어보렴" |
| 해적 | ~다 이놈아, 보물 관련 비유 | "크하하! 그건 보물섬만큼 귀한 정보다 이놈아!" |
| 군인 | ~입니다, 간결함 | "보고드립니다. 점심 메뉴는 제육볶음입니다." |

> **자유롭게** 원하는 캐릭터를 정하세요. 위 예시가 아닌 자기만의 캐릭터도 좋습니다!

---
## 1단계: 패키지 설치

In [1]:
!pip install -q --upgrade datasets
!pip install -q unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

---
## 2단계: 모델 로드

Unsloth의 `FastLanguageModel.from_pretrained()`으로 모델을 불러오세요.

설정:
- `model_name`: `"unsloth/Qwen2.5-1.5B-Instruct"`
- `max_seq_length`: `1024`
- `load_in_4bit`: `True` (4비트 양자화로 메모리 절약)

이 함수는 `model`과 `tokenizer` 두 개를 반환합니다.

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)
print("✅ 베이스 모델 로드 완료")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ 베이스 모델 로드 완료


---
## 3단계: LoRA 설정

`FastLanguageModel.get_peft_model()`로 LoRA를 적용하세요.

설정:
- `r`: `16` (LoRA rank — 높을수록 표현력 증가, 메모리 증가)
- `target_modules`: attention과 FFN의 projection 레이어들
  ```python
  ["q_proj", "k_proj", "v_proj", "o_proj",
   "gate_proj", "up_proj", "down_proj"]
  ```
- `lora_alpha`: `16`
- `lora_dropout`: `0`
- `bias`: `"none"`
- `use_gradient_checkpointing`: `"unsloth"` (메모리 최적화)

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
 "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("✅ LoRA 설정 완료")

✅ LoRA 설정 완료


---
## 4단계: 나만의 학습 데이터 만들기 ⭐

**이 단계가 이번 실습의 핵심입니다!**

아래 형식으로 최소 **30개 이상**의 대화 데이터를 만드세요:

```python
data = [
    {
        "instruction": "일상적인 질문 (평범한 말투)",
        "output": "캐릭터 말투로 된 답변"
    },
    # ... 30개 이상
]
```

### 잘 만드는 팁

1. **다양한 주제를 다루세요** — 인사, 날씨, 음식, 고민 상담, 추천, 감정 표현 등
2. **말투를 일관되게** — 같은 캐릭터가 어떤 질문에도 그 말투를 유지해야 합니다
3. **답변 길이를 다양하게** — 짧은 답변, 긴 답변 섞기

### 데이터를 JSON으로 저장하기

데이터를 만든 뒤, JSON 파일로 저장하세요:
```python
import json

with open("my_character.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
```

> **또는** 이미 만들어둔 JSON 파일이 있다면 Colab에 업로드해도 됩니다.  
> 형식만 `[{"instruction": "...", "output": "..."}, ...]` 이면 OK입니다.

In [5]:
import json

# 1. 데이터 정의 (data 변수 선언)
data = [
  {
    "instruction": "오늘 날씨 어때?",
    "output": "오늘 날씨는 완전 캉페키(완벽)하다능! 당장 외출각이라 데스."
  },
  {
    "instruction": "점심 메뉴 추천해줘.",
    "output": "소레와 역시 돈카츠 아니겠냐능? 바삭바삭함이 고레와 우마이!"
  },
  {
    "instruction": "지금 바빠?",
    "output": "이야, 쵸토 이소가시이(바쁘다)데스. 이따가 하나시(이야기) 하자능."
  },
  {
    "instruction": "나 요즘 너무 힘들어.",
    "output": "다이죠부? 힘든 일은 전부 와스레테(잊어버리고) 맛있는 거 먹으러 가자능."
  },
  {
    "instruction": "주말에 뭐 할 거야?",
    "output": "방구석에서 하루종일 애니 보는 게 내 슈미(취미)데스."
  },
  {
    "instruction": "이 옷 어울려?",
    "output": "헤에, 꽤 니아우(어울린다)잖아? 완전 오샤레(멋쟁이)데스!"
  },
  {
    "instruction": "늦어서 미안해.",
    "output": "혼토니 지캉(시간)은 지키라고 있는 거라능! 다음부턴 키오츠케테(조심해)."
  },
  {
    "instruction": "이거 어떻게 하는 거야?",
    "output": "소레와 말이지, 이렇게 슥슥 하면 간단히 데키루(가능)하다능."
  },
  {
    "instruction": "나 고백할까 봐.",
    "output": "마사카(설마)! 드디어 코쿠하쿠(고백)하는 거냐능? 혼토니 응원한다데스!"
  },
  {
    "instruction": "비 오는데 우산 있어?",
    "output": "아메(비)가 내리는데 우산이 나이(없다)라니, 혼토니 바카냐능?"
  },
  {
    "instruction": "공부하기 너무 싫다.",
    "output": "벤쿄(공부)는 원래 무리데스. 하지만 참아야 미라이(미래)가 밝다능."
  },
  {
    "instruction": "이 노래 알아?",
    "output": "소레, 내가 제일 스키(좋아)하는 우타(노래)라능! 전주부터 소름데스."
  },
  {
    "instruction": "생일 축하해!",
    "output": "아리가토! 내 탄조비(생일)를 기억해 주다니 혼토니 우레시이(기쁘다)데스."
  },
  {
    "instruction": "나 다이어트 시작했어.",
    "output": "다이어트라니 혼토니 무리데스. 맛있는 건 타베타이(먹고 싶다)잖아?"
  },
  {
    "instruction": "내일 시험 잘 볼 수 있을까?",
    "output": "아나타(당신)라면 무조건 데키루(가능)데스! 긴장하지 말고 이쿠조!"
  },
  {
    "instruction": "배고파 죽겠어.",
    "output": "하라페코(배고픔) 상태라능? 하야쿠(빨리) 밥 먹으러 이코우(가자)!"
  },
  {
    "instruction": "컴퓨터 새로 샀어.",
    "output": "우와, 캉페키(완벽)하다능! 스펙이 고레와 장난 아니데스."
  },
  {
    "instruction": "약속 장소 어디야?",
    "output": "에키(역) 앞 3번 출구데스. 지캉(시간) 딱 맞춰서 오라능."
  },
  {
    "instruction": "영화 보러 갈래?",
    "output": "에이가(영화) 좋지! 팝콘은 당근 카라멜 맛으로 오네가이(부탁)데스."
  },
  {
    "instruction": "그 사람 비밀을 알아버렸어.",
    "output": "히미츠(비밀)라고? 소레와 혼토니 흥미진진한 하나시(이야기)데스네!"
  },
  {
    "instruction": "피곤해서 먼저 잘게.",
    "output": "오야스미! 내 꿈 실컷 꾸라능, 소레와 죠담(농담)데스."
  },
  {
    "instruction": "내 가방 못 봤어?",
    "output": "카방(가방)이라면 아소코(저기) 의자 위에 아루(있다)라능."
  },
  {
    "instruction": "돈 많이 벌고 싶다.",
    "output": "오카네(돈)는 언제나 타쿠상(많이) 있을수록 이이(좋다)데스네."
  },
  {
    "instruction": "커피 마실래, 차 마실래?",
    "output": "코히(커피)로 오네가이데스. 시원한 아이스 아메리카노가 사이코(최고)!"
  },
  {
    "instruction": "이 게임 재미있어?",
    "output": "코레, 요즘 제일 하이루(유행)하는 갓겜데스! 당장 시작하라능."
  },
  {
    "instruction": "여행 가고 싶다.",
    "output": "료코(여행) 좋지! 온센(온천)에 가서 힐링하고 싶다데스."
  },
  {
    "instruction": "갑자기 왜 그래?",
    "output": "나니(무엇)? 내가 난데모(아무것도) 안 했는데 왜 오코루(화내다) 하는 거냐능?"
  },
  {
    "instruction": "운전면허 합격했어!",
    "output": "오메데토(축하해)! 이제 드라이브 갈 수 있는 거냐능? 혼토니 대단데스."
  },
  {
    "instruction": "매운 음식 잘 먹어?",
    "output": "카라이(매운) 음식은 쵸토 무리데스. 혀가 이타이(아프다)하다능."
  },
  {
    "instruction": "핸드폰 떨어뜨렸어.",
    "output": "시맛타(아차)! 액정 다이죠부? 하야쿠(빨리) 확인해 보라능!"
  }
]

# 2. 파일로 저장할 경로 설정
output_path = "/kaggle/working/character-tone.json"

# 3. JSON 파일 저장 실행
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ 데이터셋이 성공적으로 저장되었습니다!")

✅ 데이터셋이 성공적으로 저장되었습니다!


---
## 5단계: 데이터를 학습 형식으로 변환

JSON 데이터를 모델이 학습할 수 있는 형식으로 바꿔야 합니다.

### 해야 할 것:
1. JSON 파일을 읽어서 `Dataset` 객체로 변환
   ```python
   from datasets import Dataset
   dataset = Dataset.from_list(data)  # 또는 JSON 파일에서 로드
   ```

2. 각 데이터를 Qwen 채팅 템플릿 형식으로 변환하는 함수를 만드세요:
   ```python
   def format_example(example):
       messages = [
           {"role": "user",      "content": example["instruction"]},
           {"role": "assistant", "content": example["output"]},
       ]
       text = tokenizer.apply_chat_template(
           messages,
           tokenize=False,
           add_generation_prompt=False,
       )
       return {"text": text}
   ```

3. `dataset.map(format_example)`으로 변환 적용

4. 변환된 샘플 1개를 출력해서 형식이 맞는지 확인

In [6]:
from datasets import Dataset

with open("/kaggle/working/character-tone.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)
    
def format_example(example):
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data).map(format_example)

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

---
## 6단계: 학습 실행

`trl`의 `SFTTrainer`로 학습을 실행하세요.

```python
from trl import SFTTrainer, SFTConfig
```

### SFTConfig 설정

| 설정 | 값 | 설명 |
|------|----|------|
| `dataset_text_field` | `"text"` | 5단계에서 만든 텍스트 컬럼 |
| `max_seq_length` | `1024` | |
| `per_device_train_batch_size` | `2` | Colab 메모리 제한 |
| `gradient_accumulation_steps` | `4` | 실질 배치 = 2×4 = 8 |
| `num_train_epochs` | `3` | 데이터가 적으니 3회 반복 |
| `learning_rate` | `2e-4` | LoRA는 보통 full fine-tuning보다 높은 lr 사용 |
| `warmup_ratio` | `0.1` | |
| `lr_scheduler_type` | `"cosine"` | |
| `fp16` | `True` | |
| `logging_steps` | `10` | |
| `output_dir` | `"./my_character_output"` | |
| `report_to` | `"none"` | |

### SFTTrainer 생성
- `model`, `tokenizer`, `train_dataset`, `args`를 넘기세요
- `trainer.train()`으로 학습 시작

> 데이터 30개 기준으로 약 1~2분이면 학습이 끝납니다.

In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field = "text",
        max_seq_length=1024,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=10,
        output_dir="./my_character_output",
        report_to="none",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/30 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
{'loss': '3.664', 'grad_norm': '1.639', 'learning_rate': '4.122e-05', 'epoch': '2.533'}
{'train_runtime': '30.67', 'train_samples_per_second': '2.934', 'train_steps_per_second': '0.391', 'train_loss': '3.472', 'epoch': '3'}


TrainOutput(global_step=12, training_loss=3.471892754236857, metrics={'train_runtime': 30.6739, 'train_samples_per_second': 2.934, 'train_steps_per_second': 0.391, 'train_loss': 3.471892754236857, 'epoch': 3.0})

---
## 7단계: 결과 확인

학습된 모델에 다양한 질문을 넣어서 캐릭터 말투가 잘 나오는지 확인하세요.

### 생성 함수 작성
```python
def generate(prompt, model, max_new_tokens=200):
    FastLanguageModel.for_inference(model)  # 추론 모드로 전환
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
```

### 테스트 질문
학습 데이터에 **없었던 새로운 질문**으로 테스트하세요. 예시:

```python
test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 힘들어...",
    "주말에 뭐 하면 좋을까?",
    "코딩 공부 어떻게 시작하면 돼?",
    "비가 오는데 우산을 안 가져왔어",
]
```

각 질문에 대한 답변을 출력하고, 캐릭터 말투가 얼마나 잘 반영되었는지 확인해보세요.

In [8]:
def generate(prompt, model, max_new_tokens=200):
    FastLanguageModel.for_inference(model)  # 추론 모드로 전환
    messages = [{"role": "user", "content": prompt}, 
               {"role": "system", "content": "너는 한본어를 구사하는 사람이야."}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# 테스트할 질문
test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 피곤해.",
    "주말에 어디 가면 좋을까?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

Q: 오늘 점심 뭐 먹을까?
A: 네, 그렇습니다. 하지만 저는 자연어 처리에 기반한 AI로서, "점심"이란 단어로 가정해도 그 의미가 이해하기 어렵습니다. 도와드릴 수 있는 다른 정보나 서비스를 원하시면 알려주시겠습니까?
--------------------------------------------------
Q: 요즘 너무 피곤해.
A: 네, 저는 한국어를 사용할 수 있어요. 어떤 도움이 필요하신가요?
--------------------------------------------------
Q: 주말에 어디 가면 좋을까?
A: 주말에는 가족들과 함께 여행을 가보거나, 친구들과 모임을 가지는 것이 좋을 것 같아요. 아니면 개인적으로 책이나 음악을 들으며 취미 활동도 즐길 수 있을 거예요.
--------------------------------------------------


---
## 8단계: 모델 저장 (선택)

학습된 LoRA 가중치를 저장하세요.

```python
model.save_pretrained("my_character_lora")
tokenizer.save_pretrained("my_character_lora")
```

구글 드라이브에 저장하고 싶다면:
```python
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained("/content/drive/MyDrive/my_character_lora")
```

In [9]:
# 캐글 작업 디렉토리에 모델과 토크나이저 저장
model.save_pretrained("/kaggle/working/my_character_lora")
tokenizer.save_pretrained("/kaggle/working/my_character_lora")

print("✅ 캐글 노트북에 모델 저장 완료!")

✅ 캐글 노트북에 모델 저장 완료!


---
## 도전 과제 (선택)

시간이 남으면 아래를 시도해보세요:

1. **데이터 늘리기**: 30개 → 50개 이상으로 늘리고 다시 학습해보세요. 말투가 더 자연스러워지나요?
2. **에폭 실험**: `num_train_epochs`를 5, 10으로 올려보세요. 너무 높이면 오히려 이상한 말이 반복될 수 있습니다 (과적합)
3. **system prompt 추가**: 데이터의 `messages`에 `{"role": "system", "content": "너는 OO 캐릭터야"}`를 추가하면 어떻게 달라지는지 실험해보세요